# 10 — Synthèse finale de l'étude Text-to-SQL

Ce notebook est le point d'entrée de lecture des résultats finaux. Il ne relance pas les modèles : il charge les artefacts déjà produits dans `results/main_study_final`, vérifie les paramètres du protocole et construit des tableaux/figures directement exploitables dans le rapport et la soutenance.

**Protocole étudié.** 500 questions BIRD Mini-Dev, trois runs (seeds 0, 1 et 2), modèle `qwen2.5:7b`, température 0 et `top_k=5`. Les pipelines sont : A (baseline), B (RAG schéma), C (connaissance métier) et D (schéma + connaissance métier).

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'results').exists():
    ROOT = ROOT.parent
RESULTS = ROOT / 'results' / 'main_study_final'
ANALYSIS = RESULTS / 'analysis'

assert RESULTS.exists(), f'Resultats introuvables : {RESULTS}'
print(f'Racine du projet : {ROOT}')
print(f'Resultats analyses : {ANALYSIS}')

## 1. Vérification du protocole et des artefacts

Avant toute interprétation, vérifier que les quatre pipelines et les trois runs sont présents. Les scores de précision sont calculés par exécution SQLite avec une limite de 15 secondes par requête ; deux questions non comparables sont exclues conformément aux artefacts de l'étude.

In [ ]:
manifest = json.loads((RESULTS / 'main_study_manifest.json').read_text(encoding='utf-8'))
summary = json.loads((ANALYSIS / 'main_study_summary.json').read_text(encoding='utf-8'))
matrix = pd.read_csv(ANALYSIS / 'comparative_matrix.csv')

display(pd.DataFrame([manifest]))
display(matrix)

expected = {(pipeline, f'run_{run:02d}') for pipeline in 'ABCD' for run in range(3)}
observed = {(pipeline, run_name) for pipeline, data in summary['pipelines'].items()
            for run_name in data['runs']}
assert expected == observed, 'Un ou plusieurs résultats de pipeline/run sont absents.'
assert manifest['generator']['model'] == 'qwen2.5:7b'
assert manifest['generator']['temperature'] == 0.0
print('Validation réussie : 12 résultats présents ; modèle et température conformes au protocole.')

## 2. Métrique primaire : Execution Accuracy

L'**Execution Accuracy** est le pourcentage de SQL générés dont le résultat d'exécution est identique au SQL de référence. La moyenne et l'écart-type décrivent la stabilité entre les trois seeds ; ils ne remplacent pas un intervalle de confiance sur la population BIRD complète.

In [ ]:
accuracy = pd.DataFrame([
    {'pipeline': pipeline,
     'execution_accuracy_mean': values['summary']['execution_accuracy']['mean'],
     'execution_accuracy_std': values['summary']['execution_accuracy']['std'],
     'valid_sql_mean': values['summary']['valid_sql_rate']['mean'],
     'refusal_mean': values['summary']['refusal_rate']['mean']}
    for pipeline, values in summary['pipelines'].items()
])
for column in accuracy.columns[1:]:
    accuracy[column] *= 100
display(accuracy.round(2))

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(accuracy['pipeline'], accuracy['execution_accuracy_mean'],
       yerr=accuracy['execution_accuracy_std'], capsize=5, color=['#8da0cb', '#66c2a5', '#fc8d62', '#e78ac3'])
ax.set(xlabel='Pipeline', ylabel='Execution Accuracy (%)', title='Performance BIRD Mini-Dev (moyenne ± écart-type, n=3)')
ax.set_ylim(bottom=0)
ax.grid(axis='y', alpha=.25)
plt.show()

## 3. Hypothèse H1 : B contre D

H1 teste si l'ajout de connaissance métier à la récupération de schéma (D) réduit les erreurs par rapport à B, sur les mêmes questions. Le test de McNemar exact doit être interprété séparément pour les catégories **sémantique** et **structurelle**. Les trois runs sont présentés individuellement, car concaténer les mêmes questions répétées créerait des observations non indépendantes.

In [ ]:
h1_by_run = json.loads((ANALYSIS / 'h1_paired_by_run.json').read_text(encoding='utf-8'))
rows = []
for run_name, item in h1_by_run['per_run'].items():
    for kind in ('semantic_test_paired', 'structural_test_paired'):
        test = item[kind]
        rows.append({
            'run': run_name,
            'type_erreur': 'sémantique' if kind.startswith('semantic') else 'structurelle',
            'n_apparie': item['n_paired'],
            'taux_B_%': 100 * test['rate_B'],
            'taux_D_%': 100 * test['rate_D'],
            'p_McNemar': test['p_value'],
            'significatif_0.05': test['significant_at_0.05'],
        })
h1_table = pd.DataFrame(rows)
display(h1_table.round({'taux_B_%': 2, 'taux_D_%': 2, 'p_McNemar': 8}))

print('Conclusion à rapporter : D réduit fortement et significativement les erreurs sémantiques dans les trois runs.')
print('Les tests structurels run-par-run ne sont pas significatifs ; le résumé groupé montre toutefois un signal défavorable à D, à discuter comme limite.')

## 4. Annotation humaine et validité

L'annotation porte sur 60 sorties B/D échantillonnées : l'annotateur doit comparer la question, le SQL de référence, le SQL généré et le résultat d'exécution, puis choisir `structural`, `semantic`, `ambiguous`, `correct` ou `other`. Cohen's kappa mesure l'accord au-delà du hasard. Il n'est interprétable comme accord inter-annotateur que si les deux colonnes ont été renseignées indépendamment.

In [ ]:
agreement = json.loads((ANALYSIS / 'manual_annotation_agreement.json').read_text(encoding='utf-8'))
agreement_table = pd.DataFrame([{
    'cas_annotes': agreement['n_completed'],
    'accord_brut_%': 100 * agreement['raw_agreement'],
    'cohens_kappa': agreement['cohens_kappa'],
    'desaccords': len(agreement['disagreements']),
}])
display(agreement_table.round(3))
display(pd.DataFrame({'categorie': agreement['labels_annotator_a'].keys(),
                      'nombre_de_cas': agreement['labels_annotator_a'].values()}))

## 5. Conclusions, limites et livrables

- D est le meilleur pipeline sur la métrique primaire d'exécution, très devant B, C et A dans cette configuration locale.
- Les résultats appariés soutiennent un bénéfice sémantique de la connaissance métier ; ils ne permettent pas d'affirmer une amélioration structurelle.
- Les résultats concernent BIRD Mini-Dev, le modèle et le matériel/serveur local utilisés. Ils ne démontrent pas une généralisation à tous les modèles ou corpus.
- Les ablations TPC-DS et BIRD-Evidence-Corpus restent des validations secondaires du retrieval et doivent rester séparées des scores BIRD dans le mémoire.

Les fichiers à joindre au mémoire ou à la soutenance sont : `analysis/comparative_matrix.csv`, `analysis/h1_paired_by_run.json`, `analysis/manual_annotation_agreement.json`, `analysis/report_final_draft.md` et les figures dans `analysis/figures/`.